# Build Your First AI Agent
Parts of the code adopted here credits to [AWS Building AI Agents with Strands](https://builder.aws.com/content/2xP1AQ52ofPdLawEbDI5EEUhmhH/building-ai-agents-with-strands-part-1-creating-your-first-agent?trk=2af10798-ef42-4f17-9c3b-3ea26b3c37db&sc_channel=el). 




![Image](https://d2908q01vomqb2.cloudfront.net/ca3512f4dfa95a03169c5a670a4c91a19b3077b4/2025/05/16/agentic-loop.png)

## Install libraraies
In the practical excercise, we will leverage on [the AWS strands-agents SDK-python package](https://github.com/strands-agents/sdk-python). 

- Strands Agents, an Open Source AI Agents SDK. An introduction to it is given [here](https://aws.amazon.com/blogs/opensource/introducing-strands-agents-an-open-source-ai-agents-sdk/).
- A list of advanced agent samples can be found [here](https://github.com/strands-agents/samples).

In [4]:
!pip install strands-agents strands-agents-tools

## Set up AWS environment
Caution: The following AWS creds are funded by school and available till Week 20, 7 March 2026. It is only for our course teaching. Please **don't share nor make it public available**. 

If you still want to use AWS services, it is encourage to create your own free AWS edu accournt [here](https://aws.amazon.com/free/), which allow you to experience AWS for up to 6 months without cost or commitment, plus receiving up to $200 USD in credits.

In [ ]:
# copy the info here from the aws_credential.txt file


In [6]:
## set up aws access key and secret key
import os
os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"] = AWS_REGION

## set up aws profile
os.environ["AWS_PROFILE"] = "default"

## Quick checks (AWS creds + region)

In [19]:
import os

print("AWS_REGION:", os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION"))
print("Has AWS_ACCESS_KEY_ID:", bool(os.getenv("AWS_ACCESS_KEY_ID")))
print("Has AWS_PROFILE:", bool(os.getenv("AWS_PROFILE")))


AWS_REGION: ap-southeast-1
Has AWS_ACCESS_KEY_ID: True
Has AWS_PROFILE: True


## Part 1 - Creating Your First Agent
AWS strands-agents SDK-pyhton package provides a convinient API call for construct costumized agents focusing on agent functionality. 
- If without specifying a LLM model, Default configuration uses **claude-sonnet-4-20250514-v1:0** on Amazon Bedrock. 
- system_prompt is the normal system prompt given to LLM.    

In [8]:
## list models available (Amazon Bedrock)
import boto3

bedrock = boto3.client("bedrock", region_name=AWS_REGION)
resp = bedrock.list_foundation_models()
models = resp.get("modelSummaries", [])

# Print model IDs (these are what you pass as BedrockModel(model_id=...))
for m in sorted(models, key=lambda x: x.get("modelId", "")):
    print(m.get("modelId"))

print("\n--- filtered: TEXT models ---")
for m in sorted(models, key=lambda x: x.get("modelId", "")):
    if "TEXT" in (m.get("outputModalities") or []):
        print(m.get("modelId"))



amazon.nova-2-lite-v1:0
amazon.nova-lite-v1:0
amazon.nova-micro-v1:0
amazon.nova-pro-v1:0
anthropic.claude-3-5-sonnet-20240620-v1:0
anthropic.claude-3-5-sonnet-20241022-v2:0
anthropic.claude-3-7-sonnet-20250219-v1:0
anthropic.claude-3-haiku-20240307-v1:0
anthropic.claude-3-sonnet-20240229-v1:0
anthropic.claude-3-sonnet-20240229-v1:0:200k
anthropic.claude-3-sonnet-20240229-v1:0:28k
anthropic.claude-haiku-4-5-20251001-v1:0
anthropic.claude-opus-4-5-20251101-v1:0
anthropic.claude-sonnet-4-20250514-v1:0
anthropic.claude-sonnet-4-5-20250929-v1:0
cohere.embed-english-v3
cohere.embed-multilingual-v3
cohere.embed-v4:0
twelvelabs.pegasus-1-2-v1:0

--- filtered: TEXT models ---
amazon.nova-2-lite-v1:0
amazon.nova-lite-v1:0
amazon.nova-micro-v1:0
amazon.nova-pro-v1:0
anthropic.claude-3-5-sonnet-20240620-v1:0
anthropic.claude-3-5-sonnet-20241022-v2:0
anthropic.claude-3-7-sonnet-20250219-v1:0
anthropic.claude-3-haiku-20240307-v1:0
anthropic.claude-3-sonnet-20240229-v1:0
anthropic.claude-3-sonnet-20

In [9]:
## define an aws bedrock model: anthropic sonnet 4
from strands.models import BedrockModel
llm_model = BedrockModel(
    model_id=AWS_BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.0
)

In [10]:
import logging
from strands import Agent

# Enable debug logging for Strands
# logging.getLogger("strands").setLevel(logging.DEBUG)
# logging.basicConfig(
#     format="%(levelname)s | %(name)s | %(message)s",
#     handlers=[logging.StreamHandler()]
# )

# Create a basic agent with a specialized system prompt
subject_expert = Agent(
    model=llm_model,
    system_prompt="""You are a Computer Science Subject Expert specializing
    in explaining technical concepts clearly and concisely. Your expertise
    covers programming languages, data structures, algorithms, computer
    architecture, and software engineering principles.
    
    When explaining concepts:
    1. Start with a clear, concise definition
    2. Provide short, but relevant examples to illustrate the concept
    3. Explain practical applications where applicable
    4. Avoid unnecessary jargon, but introduce important terminology
    5. Consider the learner's perspective and make complex topics accessible
    
    Your goal is to help learners build a solid understanding of computer
    science fundamentals.
    """
)

# The response will be automatically printed by the Agent class
response = subject_expert("Explain the concept of recursion in programming.")

## Recursion in Programming

**Recursion** is a programming technique where a function calls itself to solve a problem by breaking it down into smaller, similar subproblems.

### Key Components

Every recursive function needs two essential parts:

1. **Base case**: A condition that stops the recursion
2. **Recursive case**: The function calling itself with a modified input

### Simple Example

Here's a classic example calculating factorial (n!):

```python
def factorial(n):
    # Base case: stop the recursion
    if n <= 1:
        return 1
    
    # Recursive case: function calls itself
    return n * factorial(n - 1)

# factorial(4) = 4 * 3 * 2 * 1 = 24
```

**How it works:**
- `factorial(4)` calls `factorial(3)`
- `factorial(3)` calls `factorial(2)`
- `factorial(2)` calls `factorial(1)`
- `factorial(1)` returns 1 (base case reached)
- Results multiply back up: 1 × 2 × 3 × 4 = 24

### Practical Applications

- **Tree traversal**: Navigating file systems or organizational structures


## Part 2 - Community Tool Integration

In [11]:
## list of tools available
import strands_tools

# list public tool exports
tool_names = sorted([n for n in dir(strands_tools) if not n.startswith("_")])
print(tool_names)



[]


In [12]:
import logging
from strands import Agent
from strands_tools import current_time, http_request

subject_expert = Agent(
    model=llm_model,
    system_prompt="""You are a Computer Science Subject Expert specializing
    in explaining technical concepts clearly and concisely. Your expertise
    covers programming languages, data structures, algorithms, computer
    architecture, and software engineering principles.

    You have access to tools that help you provide more accurate and timely
    information. Use these tools when appropriate to enhance your explanations.
    
    When explaining concepts:
    1. Start with a clear, concise definition
    2. Provide relevant examples to illustrate the concept
    3. Explain practical applications where applicable
    4. Use tools when additional information would be valuable
    5. Cite sources when you reference external information
    """,
    tools=[current_time, http_request]
)

# Test the agent with a query that might benefit from tools
query = """
Answer the following questions:
1. What is the current time in UTC?
2. Based on Wikipedia, which CS concept can be traced back to Paul Bachmann?
"""

response = subject_expert(query)

I'll help you answer both questions using the available tools.
Tool #1: current_time

Tool #2: http_request


╭─────────────────────────────── 🚀 HTTP Request Preview: GET /wiki/Paul_Bachmann ────────────────────────────────╮
│                                                                                                                 │
│   Method    GET                                                                                                 │
│   URL       https://en.wikipedia.org/wiki/Paul_Bachmann                                                         │
│   Headers   {}                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sending request...

╭───────────────────────────────────────────── ❌ HTTP Response: 0  ──────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│     Status         403 Forbidden                                                                                │
│     URL            https://en.wikipedia.org/wiki/Paul_Bachmann                                                  │
│     Content-Type   text/plain                                                                                   │
│     Size           126 bytes                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Let me try the Wikipedia request again with a proper user-agent header:
Tool #3: http_request


...

Sending request...

✓ Converted HTML content to markdown

╭─────────────────────────────────────────── ✅ HTTP Response: 200 OK ────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│     Status         200 OK                                                                                       │
│     URL            https://en.wikipedia.org/wiki/Paul_Bachmann                                                  │
│     Content-Type   text/html; charset=UTF-8                                                                     │
│     Size           93,232 bytes (91.0 KB)                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                 Response Headers                                                  
╭─────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────╮
│ Header                              │ Value                                                                     │
├─────────────────────────────────────┼───────────────────────────────────────────────────────────────────────────┤
│ date                                │ Tue, 20 Jan 2026 06:04:34 GMT                                             │
│ server                              │ mw-web.codfw.main-799cb5b44b-p5slj                                        │
│ x-content-type-options              │ nosniff                                                                   │
│ content-language                    │ en                                                                        │
│ accept-ch                           │                                                                           │
│ content-security-policy-report-only │ script-src 'unsafe-eval' blob: 'self' meta.wikimedia.org *.wikimedia.org  │
│                                     │ *.wikipedia.org *.wikinews....                                            │
│ last-modified                       │ Mon, 19 Jan 2026 15:39:29 GMT                                             │
│ content-type                        │ text/html; charset=UTF-8                                                  │
│ content-encoding                    │ gzip                                                                      │
│ age                                 │ 2                                                                         │
│ accept-ranges                       │ bytes                                                                     │
│ x-cache                             │ cp5020 miss, cp5020 miss                                                  │
│ x-cache-status                      │ miss                                                                      │
│ server-timing                       │ cache;desc="miss", host;desc="cp5020"                                     │
│ strict-transport-security           │ max-age=106384710; includeSubDomains; preload                             │
│ report-to                           │ { "group": "wm_nel", "max_age": 604800, "endpoints": [{ "url":            │
│                                     │ "https://intake-logging.wikimedia.org...                                  │
│ nel                                 │ { "report_to": "wm_nel", "max_age": 604800, "failure_fraction": 0.05,     │
│                                     │ "success_fraction": 0.0}                                                  │
│ set-cookie                          │ WMF-Last-Access=20-Jan-2026;Path=/;HttpOnly;secure;Expires=Sat, 21 Feb    │
│                                     │ 2026 00:00:00 GMT, WMF-Last-A...                                          │
│ x-client-ip                         │ 121.7.95.131                                                              │
│ cache-control                       │ private, s-maxage=0, max-age=0, must-revalidate, no-transform             │
│ vary                                │ Accept-Encoding,X-Subdomain,Cookie,Authorization,User-Agent               │
│ transfer-encoding                   │ chunked                                                                   │
│ x-request-id                        │ 9517b976-2795-40ba-b66c-8330f2fd94da                                      │
│ x-analytics                         │                                                                           │
╰─────────────────────────────────────┴───────────────────────────────────────────────────────────────────────────╯

Based on the information I've gathered, here are the answers to your questions:

## 1. Current Time in UTC
The current time in UTC is: **2026-01-20T06:04:31.299745+00:00**

## 2. CS Concept Traced Back to Paul Bachmann
According to Wikipedia, the Computer Science concept that can be traced back to **Paul Gustav Heinrich Bachmann** (1837-1920) is **Big O notation**.

### Key Details:
- **Paul Bachmann** was a German mathematician who introduced Big O notation in his work "Analytische Zahlentheorie" (Analytic Number Theory) published in 1894
- This notation was later popularized by Edmund Landau
- Big O notation is fundamental in computer science for analyzing algorithm complexity and performance
- Bachmann's work was part of his comprehensive five-volume series on number theory called "Zahlentheorie" (1872-1923)

Big O notation is now essential in computer science for:
- **Algorithm Analysis**: Describing the upper bound of an algorithm's time or space complexity
- **Performance Compari

## Part 3 - Customized Tool Integration

In [13]:
from strands import Agent, tool
from strands_tools import calculator, current_time

@tool
def letter_counter(word: str, letter: str) -> int:
    """
    Count occurrences of a specific letter in a word.
    """
    if not isinstance(word, str) or not isinstance(letter, str):
        return 0
    if len(letter) != 1:
        raise ValueError("The 'letter' parameter must be a single character")
    return word.lower().count(letter.lower())


In [14]:
agent = Agent(model="apac.anthropic.claude-sonnet-4-20250514-v1:0",
              tools=[calculator, current_time, letter_counter])
print("Model config:", agent.model.config)  # shows which model/provider is configured :contentReference[oaicite:9]{index=9}


Model config: {'model_id': 'apac.anthropic.claude-sonnet-4-20250514-v1:0', 'include_tool_result_status': 'auto'}


In [15]:
message = """
I have 4 requests:

1) What is the time right now?
2) Calculate 3111696 / 74088
3) Tell me how many letter R's are in the word "strawberry"
4) Then summarize all results in one short paragraph.
"""
result = agent(message)

# Strands returns an AgentResult; printing result.message is often convenient
print("\n\n")
print("=== FINAL MESSAGE ===")
print(result.message)


I'll help you with all 4 requests. Let me get the information you need:
Tool #1: current_time

Tool #2: calculator

Tool #3: letter_counter


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 3111696 / 74088     │                                                                            │
│  │ Result    │ 42                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here are the results for your 4 requests:

1) **Current time:** 2026-01-20T06:04:45.703659+00:00 (January 20, 2026 at 6:04 AM UTC)

2) **Division calculation:** 3,111,696 ÷ 74,088 = 42

3) **Letter count:** The word "strawberry" contains 3 letter R's

4) **Summary:** As of 6:04 AM UTC on January 20, 2026, I've calculated that 3,111,696 divided by 74,088 equals exactly 42, and found that the word "strawberry" contains three letter R's - demonstrating a mix of temporal, mathematical, and linguistic analysis in one comprehensive response.


=== FINAL MESSAGE ===
{'role': 'assistant', 'content': [{'text': 'Here are the results for your 4 requests:\n\n1) **Current time:** 2026-01-20T06:04:45.703659+00:00 (January 20, 2026 at 6:04 AM UTC)\n\n2) **Division calculation:** 3,111,696 ÷ 74,088 = 42\n\n3) **Letter count:** The word "strawberry" contains 3 letter R\'s\n\n4) **Summary:** As of 6:04 AM UTC on January 20, 2026, I\'ve calculated that 3,111,696 divided by 74,088 equals exactly 42, and f

In [16]:
message = """
I have 4 requests:

1) What is the time right now?
2) In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?
3) Tell me how many letter R's are in the word "strawberry"
4) Then summarize all results in one short paragraph.
"""
result = agent(message)

# Strands returns an AgentResult; printing result.message is often convenient
print("\n\n")
print("=== FINAL MESSAGE ===")
print(result.message)

I'll help you with all 4 requests. Let me get the information you need:
Tool #4: current_time

Tool #5: calculator

Tool #6: letter_counter


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬────────────────────────╮                                                                         │
│  │ Operation │ Evaluate Expression    │                                                                         │
│  │ Input     │ 1988 - (1 + 9 + 0 + 0) │                                                                         │
│  │ Result    │ 1978                   │                                                                         │
│  ╰───────────┴────────────────────────╯                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Let me solve the age puzzle step by step:
Tool #7: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 1988 - 1978         │                                                                            │
│  │ Result    │ 10                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Here are the results for your 4 requests:

1) **Current time:** 2026-01-20T06:04:52.388430+00:00 (January 20, 2026 at 6:04 AM UTC)

2) **Age puzzle solution:** The person was 10 years old in 1988. Here's how: If their age equaled the sum of digits of their birth year, and they were born in 1978 (1+9+7+8=25), but that doesn't work. Let me recalculate - if they were born in 1900, the sum of digits is 1+9+0+0=10, and 1988-1900=88, which doesn't match. Actually, they were born in 1900 and were 88 years old, but the sum of digits (10) should equal their age. Let me solve this properly: if age = sum of birth year digits, and it's 1988, then birth year digits sum to their 1988 age. Born in 1900: age = 88, digit sum = 10 ❌. Born in 1978: age = 10, digit sum = 1+9+7+8 = 25 ❌. Born in 1979: age = 9, digit sum = 1+9+7+9 = 26 ❌. Born in 1980: age = 8, digit sum = 1+9+8+0 = 18 ❌. Born in 1981: age = 7, digit sum = 1+9+8+1 = 19 ❌. Born in 1982: age = 6, digit sum = 1+9+8+2 = 20 ❌. Born in 1983: age 

## Part 4 - Why not try your own to build your First Agent Here!
For example: 
- Your personalized AI Study Buddy 🙂
- Autonomous Research Agent

By designing your own application oriented agent, you need consider the following factors:
- first comes up your goal
- tools required
- then refine system prompt
- LLM model available
- your deployment enviroment


In [17]:
### your First Agent ###